# Stiffness 8 — a fast reaction

This notebook repeats the first one with a different model class, on purpose.

A reversible reaction $\mathrm{A} \rightleftharpoons \mathrm{B}$ in the bulk liquid of a column that is otherwise pure transport — no particle, no binding. The two rate constants are scaled **jointly**, so $k_\mathrm{fwd}/k_\mathrm{bwd} = 4$ and the equilibrium composition never changes. Only the speed moves.

| | |
|---|---|
| column | 0.1 m, 150 cm/h, Péclet ≈ 1100, 100 cells |
| reaction | A ⇌ B in the bulk liquid, mass action |
| equilibrium | $K = 4$, so four fifths ends up as B |
| feed | 1 mol/m³ of A, 60 s pulse |

A reversible first-order reaction relaxes towards equilibrium at $k_\mathrm{fwd} + k_\mathrm{bwd}$, whatever the equilibrium constant is. That is a real, negative eigenvalue — the same kind the binding contributed in Stiffness 1. If the chapter's reading is right, the answer here should not depend on which physical process the fast mode belongs to.

In [ ]:
import utility.reaction_explorer as explorer

explorer.show_setting()

## What to watch

The **fraction converted** says when the chromatogram stops changing: once the reaction is fast enough to reach equilibrium inside the column, going faster changes nothing about the answer. Everything beyond that point is pure eigenvalue growth with the solution frozen — the clean part of the experiment.

Convection across one cell runs at 1.14 s⁻¹, so the `rate / grid rate` column tells you when the reaction has overtaken the fastest thing already in the model.

In [ ]:
from ipywidgets import interact

show_reaction = explorer.make_explorer()

interact(show_reaction, **show_reaction.sliders);

## What happened

**The same answer as the binding study, in a different model class.**

| reaction rate | fraction converted | BDF steps |
|---|---|---|
| 5·10⁻⁴ s⁻¹ | 0.034 | 2695 |
| 5·10⁻² s⁻¹ | 0.790 | 2828 |
| 5 s⁻¹ | 0.800 | 3078 |
| 5·10⁴ s⁻¹ | 0.800 | 2838 |
| 5·10¹⁰ s⁻¹ | 0.800 | **2736** |

The conversion has stopped moving by a rate of about 0.5 s⁻¹. From there to 5·10¹⁰ s⁻¹ — **eleven decades**, ending forty billion times faster than convection across a cell — the chromatogram is identical and the step count changes by **1 %**. The mean BDF order sits at 4.99 throughout.

A fast reaction is as free as a fast binding, and for the same reason: once its transient has decayed, a strongly damped real mode is carried rather than resolved. The model class does not matter; the shape of the eigenvalue does.

<details>
<summary><b>A bug found while building this notebook: quasi-stationary bulk reactions are silently ignored</b></summary>

This study was meant to end the way Stiffness 1 does, with one step past the fast limit: `MAL_IS_KINETIC = 0` replaces the rate equations by the equilibrium condition, as `IS_KINETIC = 0` does for binding.

It does not work. With `MAL_IS_KINETIC = 0` on a bulk liquid reaction, CADET runs to completion, reports success, and returns a solution in which **the reaction has not happened at all**:

| | A at the outlet | B at the outlet |
|---|---|---|
| kinetic, fast | 0.200 | 0.800 |
| quasi-stationary | **1.000** | **0.000** |

The reaction term is dropped from the dynamic residual, and no equilibrium condition is imposed in its place. There is no warning and no error — the result simply describes a column in which A passes through unchanged.

The implementation is present in `MassActionLawReaction.cpp`, which carries a separate residual path for equilibrium reactions, so this looks like that path not being reached for a reaction hosted in the bulk liquid phase of a 1D column rather than a missing feature. We have not tested a reaction in a particle phase, where the machinery for algebraic states already exists for binding.

Reported here because a silent wrong answer is worse than a crash: nothing in the output tells you the model you configured is not the model that ran.

</details>

> **Take-away.** Fast reactions cost nothing, like fast binding. Eleven decades of rate beyond the point where the chemistry has equilibrated are free, because an implicit integrator carries a damped mode instead of resolving it. If a reactive simulation is slow, the reaction rate constants are not where to look — and if you were planning to use a quasi-stationary bulk reaction to speed it up, check the chromatogram first.